[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/see-through-colab-notebook/blob/main/SeeThrough_Colab.ipynb)

# See-through: Single-image Layer Decomposition for Anime Characters

[![GitHub](https://img.shields.io/badge/GitHub-black?logo=github)](https://github.com/shitagaki-lab/see-through)
[![Paper](https://img.shields.io/badge/arXiv-2602.03749-b31b1b)](https://arxiv.org/abs/2602.03749)
[![License](https://img.shields.io/badge/License-Apache_2.0-green)](https://github.com/shitagaki-lab/see-through/blob/main/LICENSE)

Published at **ACM SIGGRAPH 2026**. Decomposes a single anime illustration into up to **23 semantic transparent layers** (hair, face, eyes, clothing, accessories…) with depth ordering, exported as a layered **PSD** file.

| Model | Repo | Size |
|---|---|---|
| LayerDiff 3D | `layerdifforg/seethroughv0.0.2_layerdiff3d` | 4B params |
| Marigold Depth | `layerdifforg/seethroughv0.0.1_marigold` | 1B params |
| NF4 LayerDiff | `24yearsold/seethroughv0.0.2_layerdiff3d_nf4` | ~2B params (4-bit) |
| NF4 Marigold | `24yearsold/seethroughv0.0.1_marigold_nf4` | ~0.5B params (4-bit) |

**VRAM**: bf16 needs ~12–16 GB at 1280 res. NF4 quantized runs at ~8 GB peak.

**Persistence**: Models cached to GDrive (`SeeThrough/models/`), outputs saved to GDrive (`SeeThrough/outputs/`).

Run all cells top-to-bottom. A Gradio Web UI link will appear at the end.

## 1 · Environment Setup

In [ ]:
#@title 1.1 Check GPU
#@markdown This notebook requires a GPU. Go to **Runtime → Change runtime type → T4 GPU**.
import torch, sys

if not torch.cuda.is_available():
    print("\u274c No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\u2705 GPU: {gpu_name} ({vram_gb:.1f} GB)")

# bf16 needs ~12-16 GB for both pipelines simultaneously on GPU.
# Colab T4 (16GB) OOMs with bf16 — force NF4 to be safe.
USE_NF4 = True  # NF4: ~5 GB total, safe on any Colab GPU
print(f"   NF4 quantization: ON (safe default for Colab)")
print(f"   Tip: set USE_NF4 = False below only if you have >20 GB VRAM.")


In [ ]:
#@title 1.2 Mount Google Drive (persistent models & outputs)
#@markdown First run asks for auth. After that, models (~10 GB) persist across sessions.
import os, pathlib

GDRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/SeeThrough")
GDRIVE_MODELS = GDRIVE_ROOT / "models"
GDRIVE_OUTPUTS = GDRIVE_ROOT / "outputs"

if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
    print("\u2705 Google Drive mounted.")
else:
    print("\u2705 Google Drive already mounted.")

# Create persistent directories
GDRIVE_MODELS.mkdir(parents=True, exist_ok=True)
GDRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

# Set HuggingFace cache to GDrive so models survive runtime resets
os.environ["HF_HOME"] = str(GDRIVE_MODELS / "hf_cache")
os.environ["TRANSFORMERS_CACHE"] = str(GDRIVE_MODELS / "hf_cache")
os.environ["HF_HUB_CACHE"] = str(GDRIVE_MODELS / "hf_cache")

# Set torch hub cache too
os.environ["TORCH_HOME"] = str(GDRIVE_MODELS / "torch")

print(f"   Models cache:  {GDRIVE_MODELS}")
print(f"   Outputs dir:  {GDRIVE_OUTPUTS}")

# Report existing cache size
hf_cache = GDRIVE_MODELS / "hf_cache"
if hf_cache.exists():
    total = sum(f.stat().st_size for f in hf_cache.rglob("*") if f.is_file())
    print(f"   Cached models: {total / (1024**3):.1f} GB")
else:
    print("   No cached models yet (first run will download).")

In [ ]:
#@title 1.3 Clone See-through repo & install dependencies
import os

REPO_DIR = "/content/see-through"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/shitagaki-lab/see-through.git {REPO_DIR}
else:
    print("Repo already cloned.")

# Install core deps (skip torch/torchvision — Colab already has them)
!pip install -q \
    diffusers==0.37.0 \
    transformers>=5.0.0 \
    accelerate>=1.13.0 \
    safetensors>=0.7.0 \
    huggingface-hub>=1.7.2 \
    einops>=0.8.2 \
    opencv-python-headless \
    psd-tools[composite] \
    scikit-learn \
    matplotlib \
    pyyaml \
    tqdm \
    pycocotools \
    gradio>=4.0.0

# NF4 quantization (only if needed)
if USE_NF4:
    !pip install -q bitsandbytes

print("\u2705 Dependencies installed.")

In [ ]:
#@title 1.4 Add repo to Python path & symlink assets
import sys, os

REPO_DIR = "/content/see-through"
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "common"))

# Symlink assets (test images etc.)
assets_link = os.path.join(REPO_DIR, "assets")
common_assets = os.path.join(REPO_DIR, "common", "assets")
if not os.path.exists(assets_link) and os.path.exists(common_assets):
    os.symlink(common_assets, assets_link)

print(f"\u2705 Python path includes {REPO_DIR}")
print(f"   Assets: {assets_link} -> {common_assets if os.path.islink(assets_link) else 'exists'}")

## 2 · Load Models

In [ ]:
#@title 2.1 Load LayerDiff 3D + Marigold pipelines
#@markdown Models are cached in GDrive. First run downloads ~5 GB (NF4).
#@markdown Subsequent sessions load from cache instantly.
import torch, time, os, pathlib, gc

# Fallback if cell 1.1 was skipped
if 'USE_NF4' not in dir():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    USE_NF4 = True
    print(f'Auto-detected VRAM: {vram_gb:.1f} GB \u2192 NF4: ON')

REPO_LAYERDIFF_BF16 = "layerdifforg/seethroughv0.0.2_layerdiff3d"
REPO_DEPTH_BF16     = "24yearsold/seethroughv0.0.1_marigold"
REPO_LAYERDIFF_NF4  = "24yearsold/seethroughv0.0.2_layerdiff3d_nf4"
REPO_DEPTH_NF4      = "24yearsold/seethroughv0.0.1_marigold_nf4"

REPO_LAYERDIFF = REPO_LAYERDIFF_NF4 if USE_NF4 else REPO_LAYERDIFF_BF16
REPO_DEPTH     = REPO_DEPTH_NF4     if USE_NF4 else REPO_DEPTH_BF16


def move_to_gpu(pipe, use_nf4):
    """Move pipeline components to GPU safely.
    NF4 quantized models: device only, no dtype cast.
    bf16 models: device + dtype=bfloat16.
    Avoids .to() on the whole pipeline (breaks on some internal modules)."""
    dtype = torch.bfloat16 if not use_nf4 else None
    for name in ['vae', 'trans_vae']:
        m = getattr(pipe, name, None)
        if m is not None:
            if dtype and not getattr(m, 'is_quantized', False):
                m.to(dtype=dtype, device='cuda')
            else:
                m.to(device='cuda')
    unet = getattr(pipe, 'unet', None)
    if unet is not None:
        if dtype and not getattr(unet, 'is_quantized', False):
            unet.to(dtype=dtype, device='cuda')
        else:
            unet.to(device='cuda')
    for name in ['text_encoder', 'text_encoder_2']:
        m = getattr(pipe, name, None)
        if m is not None:
            if dtype and not getattr(m, 'is_quantized', False):
                m.to(dtype=dtype, device='cuda')
            else:
                m.to(device='cuda')


# ---- Load LayerDiff ----
print(f"Loading LayerDiff from: {REPO_LAYERDIFF}")
print(f"   Cache: {os.environ.get('HF_HOME', 'default')}")
t0 = time.time()

from modules.layerdiffuse.diffusers_kdiffusion_sdxl import KDiffusionStableDiffusionXLPipeline
from modules.layerdiffuse.layerdiff3d import UNetFrameConditionModel
from modules.layerdiffuse.vae import TransparentVAE

trans_vae = TransparentVAE.from_pretrained(REPO_LAYERDIFF, subfolder="trans_vae")
unet_ld   = UNetFrameConditionModel.from_pretrained(REPO_LAYERDIFF, subfolder="unet")
layerdiff_pipe = KDiffusionStableDiffusionXLPipeline.from_pretrained(
    REPO_LAYERDIFF, trans_vae=trans_vae, unet=unet_ld, scheduler=None
)
print(f"   LayerDiff loaded ({time.time()-t0:.1f}s)")

# Cache tag embeddings on CPU before moving to GPU
print("Caching LayerDiff tag embeddings (on CPU)...")
layerdiff_pipe.cache_tag_embeds()

# Move to GPU
print("Moving LayerDiff to GPU...")
t0 = time.time()
move_to_gpu(layerdiff_pipe, USE_NF4)
print(f"   LayerDiff on GPU ({time.time()-t0:.1f}s)")

# Free text encoders from CPU (tag embeds already cached)
for attr in ['text_encoder', 'text_encoder_2']:
    if hasattr(layerdiff_pipe, attr):
        delattr(layerdiff_pipe, attr)
gc.collect()
print("   Text encoders freed.")

# ---- Load Marigold ----
print(f"Loading Marigold from: {REPO_DEPTH}")
t0 = time.time()

from modules.marigold import MarigoldDepthPipeline

unet_mg = UNetFrameConditionModel.from_pretrained(REPO_DEPTH, subfolder="unet")
marigold_pipe = MarigoldDepthPipeline.from_pretrained(REPO_DEPTH, unet=unet_mg)
print(f"   Marigold loaded ({time.time()-t0:.1f}s)")

# Cache tag embeddings on CPU before moving
print("Caching Marigold tag embeddings (on CPU)...")
marigold_pipe.cache_tag_embeds()

# Move to GPU
print("Moving Marigold to GPU...")
t0 = time.time()
move_to_gpu(marigold_pipe, USE_NF4)
print(f"   Marigold on GPU ({time.time()-t0:.1f}s)")

# Free text encoder
if hasattr(marigold_pipe, 'text_encoder'):
    delattr(marigold_pipe, 'text_encoder')
gc.collect()
torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / (1024**3)
reserved  = torch.cuda.memory_reserved()  / (1024**3)
print(f"\u2705 All models loaded. GPU: {allocated:.1f} GB alloc / {reserved:.1f} GB reserved")


## 3 · Inference Function

In [ ]:
#@title 3.1 Define decomposition pipeline
#@markdown Outputs are saved to `GDrive/SeeThrough/outputs/` for persistence.
import os, sys, time, tempfile, shutil
from PIL import Image

from utils.inference_utils import apply_layerdiff, apply_marigold, further_extr
from utils.torch_utils import seed_everything
import utils.inference_utils as _inf

# Inject loaded pipelines so apply_* functions skip their own loading
_inf.layerdiff_pipeline = layerdiff_pipe
_inf.marigold_pipeline  = marigold_pipe

_SKIP_TAGS = {"src_img", "src_head", "reconstruction"}


def collect_layer_gallery(saved_dir):
    """Collect layer PNGs as (image, label) tuples for Gradio gallery."""
    gallery = []
    for f in sorted(os.listdir(saved_dir)):
        if not f.endswith(".png"):
            continue
        tag = f[:-4]
        if tag.endswith("_depth") or tag in _SKIP_TAGS:
            continue
        img = Image.open(os.path.join(saved_dir, f))
        gallery.append((img, tag))
    return gallery


def decompose(image: Image.Image, resolution: int = 768, seed: int = 42,
              tblr_split: bool = False):
    """Full See-through pipeline: image -> layered PSD + layer gallery."""
    if image is None:
        raise ValueError("Please upload an image.")

    resolution = max(64, min(resolution, 1280))
    resolution = round(resolution / 64) * 64

    seed_everything(seed)
    tmpdir = tempfile.mkdtemp(prefix="seethrough_")

    try:
        input_path = os.path.join(tmpdir, "input.png")
        image.save(input_path)

        t_start = time.time()

        print(f"[LayerDiff] resolution={resolution} seed={seed}")
        t0 = time.time()
        apply_layerdiff(input_path, REPO_LAYERDIFF,
                        save_dir=tmpdir, seed=seed, resolution=resolution)
        print(f"[LayerDiff] done ({time.time()-t0:.1f}s)")

        t0 = time.time()
        apply_marigold(input_path, REPO_DEPTH,
                       save_dir=tmpdir, seed=seed, resolution=resolution)
        print(f"[Marigold] done ({time.time()-t0:.1f}s)")

        saved = os.path.join(tmpdir, "input")
        gallery = collect_layer_gallery(saved)

        t0 = time.time()
        further_extr(saved, rotate=False, save_to_psd=True, tblr_split=tblr_split)
        print(f"[PSD] done ({time.time()-t0:.1f}s)")

        psd_path = saved + ".psd"
        if not os.path.exists(psd_path):
            raise RuntimeError("PSD generation failed.")

        # ---- Save to GDrive for persistence ----
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        psd_name = f"see_through_{timestamp}_s{seed}.psd"
        out_psd = str(GDRIVE_OUTPUTS / psd_name)
        shutil.copy2(psd_path, out_psd)

        # Save layer PNGs to GDrive too
        out_layers_dir = GDRIVE_OUTPUTS / f"layers_{timestamp}_s{seed}"
        if out_layers_dir.exists():
            shutil.rmtree(out_layers_dir)
        shutil.copytree(saved, str(out_layers_dir))

        print(f"\u2705 Total: {time.time()-t_start:.1f}s | {len(gallery)} layers")
        print(f"   Saved: {out_psd}")
        return out_psd, gallery

    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


print("\u2705 Inference function ready. Outputs will persist in GDrive.")

## 4 · Web UI

Choose a tunnel method:

| Method | Pros | Cons |
|---|---|---|
| **Gradio share** | Zero config, works immediately | 72h expiry, goes through Gradio relay |
| **ngrok** | Stable, low latency, custom domains | Needs free auth token from ngrok.com |

In [ ]:
#@title 4.1 Launch Web UI
#@markdown **Tunnel method**: choose Gradio (simple) or ngrok (stable).
#@markdown **ngrok auth token** (optional): get free at https://dashboard.ngrok.com/get-started/your-authtoken
import gradio as gr

LAUNCH_METHOD = "Gradio share"  #@param ["Gradio share", "ngrok"]
NGROK_AUTH_TOKEN = ""  #@param {type:"string"}

# Build the Gradio app
with gr.Blocks(title="See-through: Layer Decomposition") as demo:
    gr.Markdown(
        "# See-through: Single-image Layer Decomposition\n\n"
        "Upload an anime character illustration to decompose it into "
        "fully-inpainted semantic layers with depth ordering, "
        "exported as a layered PSD file.\n\n"
        f"**Model**: {REPO_LAYERDIFF.split('/')[-1]} | "
        f"**Precision**: {'NF4 (4-bit)' if USE_NF4 else 'bf16'} | "
        f"**VRAM**: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB\n\n"
        f"\U0001f4be **Outputs persist in**: `{GDRIVE_OUTPUTS}`"
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(
                type="pil",
                label="Upload anime illustration",
                sources=["upload", "clipboard"],
            )
            resolution = gr.Slider(
                minimum=768, maximum=1280, value=768, step=64,
                label="Resolution",
                info="768 recommended. Higher = better quality, more VRAM & time.",
            )
            seed = gr.Slider(
                minimum=0, maximum=9999, value=42, step=1,
                label="Seed",
            )
            tblr_split = gr.Checkbox(
                value=False,
                label="Split left/right arms & legs",
                info="Separate symmetric parts into individual layers.",
            )
            run_btn = gr.Button("\u25b6 Run Decomposition", variant="primary", size="lg")

        with gr.Column(scale=2):
            psd_output = gr.File(label="\u2b07 Download Layered PSD")
            gallery_output = gr.Gallery(
                label="Separated Layers",
                columns=4,
                height="auto",
                object_fit="contain",
            )

    run_btn.click(
        fn=decompose,
        inputs=[input_image, resolution, seed, tblr_split],
        outputs=[psd_output, gallery_output],
        api_name="decompose",
    )

    gr.Examples(
        examples=[
            ["/content/see-through/common/assets/test_image.png", 768, 42, False],
        ],
        inputs=[input_image, resolution, seed, tblr_split],
        label="Example",
    )

# Launch with chosen method
if LAUNCH_METHOD == "ngrok":
    # ngrok tunnel: stable, low latency, no 72h expiry
    !pip install -q pyngrok
    from pyngrok import ngrok, conf

    if NGROK_AUTH_TOKEN:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    else:
        print("\u26a0\ufe0f  No ngrok token — using limited anonymous tunnel (may hit rate limits).")
        print("   Get a free token: https://dashboard.ngrok.com/get-started/your-authtoken")

    # Kill any existing ngrok tunnels
    ngrok.kill()

    # Start Gradio server (no share tunnel)
    app, local_url, _ = demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        prevent_thread_cleanup=True,
        theme=gr.themes.Soft(),
    )

    # Create ngrok tunnel to 7860
    public_url = ngrok.connect(7860, "http")
    print(f"\n\u2705 ngrok tunnel active!")
    print(f"   Public URL: {public_url}")
    print(f"   Local URL:  {local_url}")
    print(f"   Tunnel persists until you stop the runtime.")
else:
    # Gradio share: zero config, simplest
    demo.launch(share=True, debug=True, theme=gr.themes.Soft())


## 5 · (Optional) Batch Processing

In [ ]:
#@title 5.1 Process a folder of images
#@markdown Upload images to a folder, set the path, and run.
#@markdown Outputs auto-save to `GDrive/SeeThrough/outputs/`.
#@markdown ```
#@markdown /content/my_images/
#@markdown   ├── char1.png
#@markdown   ├── char2.png
#@markdown   └── ...
#@markdown ```
import os, glob

BATCH_DIR = "/content/my_images"  #@param {type:"string"}
BATCH_RESOLUTION = 768  #@param {type:"integer"}
BATCH_SEED = 42  #@param {type:"integer"}
BATCH_TLBR_SPLIT = False  #@param {type:"boolean"}

exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
images = sorted([
    f for f in glob.glob(os.path.join(BATCH_DIR, "*"))
    if os.path.splitext(f)[1].lower() in exts
])

print(f"Found {len(images)} images in {BATCH_DIR}")
print(f"Output dir: {GDRIVE_OUTPUTS}")

for i, img_path in enumerate(images):
    name = os.path.splitext(os.path.basename(img_path))[0]
    # Check if PSD already exists in GDrive outputs
    existing = list(GDRIVE_OUTPUTS.glob(f"*{name}*.psd"))
    if existing:
        print(f"[{i+1}/{len(images)}] SKIP {name} (PSD exists)")
        continue

    print(f"\n[{i+1}/{len(images)}] Processing: {name}")
    try:
        img = Image.open(img_path).convert("RGBA")
        psd_path, gallery = decompose(
            img, resolution=BATCH_RESOLUTION, seed=BATCH_SEED,
            tblr_split=BATCH_TLBR_SPLIT,
        )
    except Exception as e:
        print(f"  \u274c Error: {e}")

print(f"\n\u2705 Batch done. All outputs in: {GDRIVE_OUTPUTS}")

## Troubleshooting

| Problem | Fix |
|---|---|
| `No GPU detected` | Runtime → Change runtime type → T4 GPU |
| `CUDA out of memory` | Lower resolution to 768; ensure NF4 is ON for <12 GB VRAM |
| `GDrive auth failed` | Re-run cell 1.2; check you're logged into the right Google account |
| Download slow (first run) | Models cache to GDrive; subsequent sessions are instant |
| `ModuleNotFoundError` | Re-run cell 1.4 (path setup) |
| Gradio share URL expired | Re-run cell 4.1; or switch to ngrok for stable tunnel |
| ngrok rate limit | Add auth token in cell 4.1 (free at ngrok.com) |
| ngrok connection refused | Ensure cell 2.1 (model loading) completed before launching |

### GDrive Structure

```
MyDrive/SeeThrough/
├── models/
│   └── hf_cache/          ← ~10 GB model weights (cached)
└── outputs/
    ├── see_through_20260928_120000_s42.psd
    ├── layers_20260928_120000_s42/
    │   ├── hair.png
    │   ├── face.png
    │   └── ...
    └── ...
```

### References

- **Paper**: [arXiv:2602.03749](https://arxiv.org/abs/2602.03749) (SIGGRAPH 2026)
- **Code**: [github.com/shitagaki-lab/see-through](https://github.com/shitagaki-lab/see-through)
- **ComfyUI**: [github.com/jtydhr88/ComfyUI-See-through](https://github.com/jtydhr88/ComfyUI-See-through)
- **HF Space**: [huggingface.co/spaces/24yearsold/see-through-demo](https://huggingface.co/spaces/24yearsold/see-through-demo)